# Experiment: SpectralBio Block 2 - Failure Mode Hunt (H100)

Objective:
- Turn the current covariance-heavy regime into a reviewer-facing failure mode.
- Reuse the strongest frozen evidence first, then run bounded H100 work only if needed.
- Annotate the regime biologically enough to support a simple bug statement.
- Save a complete Lightning-friendly bundle under `New Notebooks/results/02_block2_failure_mode_hunt_h100/`.


## Block 2 Deliverables

- Reused-or-rerun validation rows for the candidate regime.
- A ranked exemplar table for strong covariance repair without likelihood repair.
- Context annotations: substitution chemistry, AlphaFold pLDDT when available, and UniProt feature overlap.
- Enrichment tables against matched controls, benign regime rows, and non-repair candidates.
- A reviewer-facing `bug_statement` that stays honest if the biology is only suggestive.

## Runtime contract

- Target environment: **Lightning AI Studio**
- Target hardware: **H100**
- Heavy rescoring is optional and gated.
- All outputs are written under `New Notebooks/results/02_block2_failure_mode_hunt_h100/`.


In [ ]:
# Setup: imports, identifiers, and notebook knobs
from __future__ import annotations

import importlib
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import urllib.error
import urllib.parse
import urllib.request
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import fisher_exact

NOTEBOOK_SLUG = '02_block2_failure_mode_hunt_h100'
ACCOUNT_LABEL = os.environ.get('SPECTRALBIO_ACCOUNT_LABEL', 'SET_ACCOUNT_LABEL_HERE')
RUN_AT = datetime.now(timezone.utc).isoformat()
SEED = int(os.environ.get('SPECTRALBIO_RANDOM_SEED', '42'))
FORCE_HEAVY = os.environ.get('SPECTRALBIO_BLOCK2_FORCE_HEAVY', '').strip().lower() in {'1', 'true', 'yes'}
OVERWRITE = os.environ.get('SPECTRALBIO_OVERWRITE', '').strip().lower() in {'1', 'true', 'yes'}
FROB_REPAIR_MIN = float(os.environ.get('SPECTRALBIO_BLOCK2_FROB_REPAIR_MIN', '0.25'))
LL_REPAIR_MAX = float(os.environ.get('SPECTRALBIO_BLOCK2_LL_REPAIR_MAX', '0.10'))
PAIR_REPAIR_MIN = float(os.environ.get('SPECTRALBIO_BLOCK2_PAIR_REPAIR_MIN', '0.00'))
MIN_CONTEXT_COUNT = int(os.environ.get('SPECTRALBIO_BLOCK2_MIN_CONTEXT_COUNT', '2'))
REQUEST_TIMEOUT_SEC = int(os.environ.get('SPECTRALBIO_BLOCK2_REQUEST_TIMEOUT_SEC', '30'))

AA_CHARGE = {
    'D': 'acidic', 'E': 'acidic',
    'K': 'basic', 'R': 'basic', 'H': 'basic',
    'S': 'polar', 'T': 'polar', 'N': 'polar', 'Q': 'polar',
    'A': 'hydrophobic', 'V': 'hydrophobic', 'L': 'hydrophobic', 'I': 'hydrophobic', 'M': 'hydrophobic',
    'F': 'aromatic', 'W': 'aromatic', 'Y': 'aromatic',
    'G': 'special', 'P': 'special', 'C': 'special',
}
AA_VOLUME = {
    'G': 1, 'A': 2, 'S': 2, 'C': 2, 'P': 3, 'T': 3, 'D': 3, 'N': 3,
    'V': 4, 'E': 4, 'Q': 4, 'H': 4, 'I': 4, 'L': 4, 'M': 4, 'K': 5,
    'R': 6, 'F': 6, 'Y': 6, 'W': 7,
}
CONFIDENT_PLDDT = 70.0
VERY_HIGH_PLDDT = 90.0
DISORDER_PLDDT = 50.0

def done(message: str) -> None:
    print(f'TERMINEI PODE SEGUIR - {message}')

print({
    'notebook_slug': NOTEBOOK_SLUG,
    'account_label': ACCOUNT_LABEL,
    'seed': SEED,
    'force_heavy': FORCE_HEAVY,
    'overwrite': OVERWRITE,
    'frob_repair_min': FROB_REPAIR_MIN,
    'll_repair_max': LL_REPAIR_MAX,
    'pair_repair_min': PAIR_REPAIR_MIN,
    'min_context_count': MIN_CONTEXT_COUNT,
    'python': sys.version.split()[0],
    'platform': platform.platform(),
})
done('Initial configuration loaded.')


In [ ]:
# Helpers: subprocess, repo discovery, paths, HTTP, and tabular utilities
def run(command: list[str], cwd: Path | None = None, check: bool = True) -> str:
    completed = subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        text=True,
        encoding='utf-8',
        errors='replace',
        capture_output=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return completed.stdout

def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()

def find_repo_root() -> Path:
    env_repo = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()
    candidates: list[Path] = []
    if env_repo:
        candidates.append(Path(env_repo).expanduser())
    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        Path('/teamspace/studios/this_studio/Stanford-Claw4s'),
        Path('/teamspace/studios/this_studio/SpectralBio'),
        Path('/content/Stanford-Claw4s'),
    ])
    for candidate in candidates:
        if looks_like_repo(candidate):
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the SpectralBio repository root. Set SPECTRALBIO_REPO_ROOT or run inside the repo.')

def resolve_existing_path(raw: str | Path, repo_root: Path) -> Path:
    raw_path = Path(raw)
    if raw_path.exists():
        return raw_path.resolve()
    raw_text = str(raw).replace('\\', '/')
    for prefix in ('/content/Stanford-Claw4s/', '/teamspace/studios/this_studio/Stanford-Claw4s/', '/teamspace/studios/this_studio/SpectralBio/'):
        if raw_text.startswith(prefix):
            candidate = repo_root / raw_text[len(prefix):]
            if candidate.exists():
                return candidate.resolve()
    if 'Stanford-Claw4s/' in raw_text:
        candidate = repo_root / raw_text.split('Stanford-Claw4s/', 1)[1]
        if candidate.exists():
            return candidate.resolve()
    if not raw_path.is_absolute():
        candidate = (repo_root / raw_path).resolve()
        if candidate.exists():
            return candidate
    return raw_path

def find_first_existing_path(candidates: list[str | Path], repo_root: Path) -> Path | None:
    for candidate in candidates:
        resolved = resolve_existing_path(candidate, repo_root)
        if resolved.exists():
            return resolved
    return None

def fetch_json(url: str) -> dict | list | None:
    request = urllib.request.Request(url, headers={'User-Agent': 'SpectralBio-Block2/1.0'})
    try:
        with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SEC) as response:
            return json.loads(response.read().decode('utf-8'))
    except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, json.JSONDecodeError) as exc:
        print(f'HTTP warning for {url}: {type(exc).__name__}: {exc}')
        return None

def minmax_normalize(series: pd.Series) -> pd.Series:
    values = series.astype(float)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values), dtype=float), index=series.index)
    return (values - minimum) / (maximum - minimum)

def coerce_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {'1', 'true', 'yes', 'y'}

def safe_float(value, default: float = float('nan')) -> float:
    try:
        return float(value)
    except Exception:
        return default

done('Helpers ready.')


In [ ]:
# Resolve repository root and notebook result directories
REPO_ROOT = find_repo_root()
RESULTS_ROOT = ensure_dir(REPO_ROOT / 'New Notebooks' / 'results' / NOTEBOOK_SLUG)
TABLES_DIR = ensure_dir(RESULTS_ROOT / 'tables')
FIGURES_DIR = ensure_dir(RESULTS_ROOT / 'figures')
TEXT_DIR = ensure_dir(RESULTS_ROOT / 'text')
MANIFESTS_DIR = ensure_dir(RESULTS_ROOT / 'manifests')
RUNTIME_DIR = ensure_dir(RESULTS_ROOT / 'runtime')
ZIP_PATH = REPO_ROOT / 'New Notebooks' / 'results' / f'{NOTEBOOK_SLUG}.zip'
ROOT_ZIP_COPY = REPO_ROOT / 'New Notebooks' / f'{NOTEBOOK_SLUG}.zip'
repo_status = {
    'repo_root': str(REPO_ROOT),
    'head_commit': run(['git', 'rev-parse', 'HEAD'], cwd=REPO_ROOT).strip(),
    'head_subject': run(['git', 'log', '-1', '--pretty=%s'], cwd=REPO_ROOT).strip(),
    'branch': run(['git', 'branch', '--show-current'], cwd=REPO_ROOT).strip(),
}
display(pd.DataFrame([repo_status]))
done('Repository and output directories confirmed.')


In [ ]:
# Lightning preflight: install runtime dependencies in the active kernel and clear stale import state when needed
RUNTIME_REQUIREMENTS = [
    ('numpy', 'numpy==2.1.3'),
    ('pandas', 'pandas==2.2.3'),
    ('matplotlib', 'matplotlib==3.9.2'),
    ('scipy', 'scipy==1.14.1'),
    ('sklearn', 'scikit-learn==1.5.2'),
    ('torch', 'torch'),
    ('transformers', 'transformers==4.49.0'),
    ('accelerate', 'accelerate>=1.0.0'),
    ('sentencepiece', 'sentencepiece>=0.2.0'),
    ('google.protobuf', 'protobuf>=5'),
]
runtime_rows = []
missing_packages: list[str] = []
for module_name, package_spec in RUNTIME_REQUIREMENTS:
    try:
        importlib.import_module(module_name)
        runtime_rows.append({'module': module_name, 'status': 'ok', 'package_spec': package_spec})
    except Exception as exc:
        runtime_rows.append({'module': module_name, 'status': f'missing:{type(exc).__name__}', 'package_spec': package_spec})
        missing_packages.append(package_spec)
display(pd.DataFrame(runtime_rows))
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', *missing_packages])
    modules_to_drop = [
        name for name in list(sys.modules)
        if name == 'accelerate'
        or name.startswith('accelerate.')
        or name == 'transformers'
        or name.startswith('transformers.')
        or name == 'spectralbio'
        or name.startswith('spectralbio.')
    ]
    for name in modules_to_drop:
        sys.modules.pop(name, None)
    importlib.invalidate_caches()
    print('Installed runtime packages:', ', '.join(missing_packages))
import accelerate
import torch
from transformers.utils.import_utils import is_accelerate_available
runtime_manifest = {
    'python': sys.version.split()[0],
    'torch_version': getattr(torch, '__version__', 'unknown'),
    'accelerate_version': getattr(accelerate, '__version__', 'unknown'),
    'cuda_available': bool(torch.cuda.is_available()),
    'cuda_device_count': int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
    'transformers_sees_accelerate': bool(is_accelerate_available()),
    'account_label': ACCOUNT_LABEL,
    'run_at': RUN_AT,
}
(RUNTIME_DIR / 'runtime_manifest.json').write_text(json.dumps(runtime_manifest, indent=2), encoding='utf-8')
display(pd.DataFrame([runtime_manifest]))
done('Runtime dependencies are ready for Lightning.')


In [ ]:
# Import local SpectralBio helpers after runtime preflight
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=str(REPO_ROOT), check=False)
from spectralbio.constants import WINDOW_RADIUS
from spectralbio.supplementary.final_accept_hardening import AcceptHardeningConfig, create_accept_hardening_paths

CONFIG = AcceptHardeningConfig(
    overwrite=OVERWRITE,
    checkpoint_every=1,
    window_radius=WINDOW_RADIUS,
)
PATHS = create_accept_hardening_paths(REPO_ROOT)
done('Local package imports are clean inside Lightning.')


In [ ]:
# Locate frozen evidence first; Block 2 should reuse strong evidence whenever possible
FROZEN_EVIDENCE_CANDIDATES = {
    'screen_summary': [
        REPO_ROOT / 'supplementary' / 'failure_mode_screen_t4' / 'failure_mode_screen' / 'tables' / 'candidate_failure_mode_summary.json',
        REPO_ROOT / 'notebooks' / 'Results 7,8,9' / 'failure_mode_screen_bundle' / 'tables' / 'candidate_failure_mode_summary.json',
    ],
    'validation_rows': [
        REPO_ROOT / 'supplementary' / 'failure_mode_validation_h100' / 'failure_mode_validation' / 'tables' / 'failure_mode_validation_rows.csv',
        REPO_ROOT / 'notebooks' / 'Results 7,8,9' / 'failure_mode_validation_bundle' / 'tables' / 'failure_mode_validation_rows.csv',
        REPO_ROOT / 'notebooks' / 'Results 7,8,9' / 'failure_mode_scale_repair_bundle8B' / 'failure_mode_scale_repair_bundle' / 'tables' / 'failure_mode_scale_repair_rows.csv',
    ],
    'scale_repair_summary': [
        REPO_ROOT / 'notebooks' / 'Results 7,8,9' / 'failure_mode_scale_repair_bundle8B' / 'failure_mode_scale_repair_bundle' / 'tables' / 'failure_mode_scale_repair_summary.json',
    ],
    'robustness_summary': [
        REPO_ROOT / 'notebooks' / 'Results 7,8,9' / 'failure_mode_robustness_audit_bundle10' / 'tables' / 'failure_mode_robustness_summary.json',
    ],
    'sister_summary': [
        REPO_ROOT / 'notebooks' / 'results 11' / 'failure_mode_sister_substitution_audit_bundle11' / 'tables' / 'sister_substitution_summary.json',
    ],
}
artifact_rows: list[dict] = []
resolved_artifacts: dict[str, Path | None] = {}
for key, candidates in FROZEN_EVIDENCE_CANDIDATES.items():
    resolved = find_first_existing_path(candidates, REPO_ROOT)
    resolved_artifacts[key] = resolved
    artifact_rows.append({
        'artifact_key': key,
        'resolved_path': str(resolved) if resolved is not None else '',
        'exists': resolved is not None,
    })
artifact_inventory = pd.DataFrame(artifact_rows)
artifact_inventory.to_csv(TABLES_DIR / 'frozen_evidence_inventory.csv', index=False)
display(artifact_inventory)
done('Frozen evidence inventory saved.')


In [ ]:
# Reuse validation rows if available; fall back to a bounded rerun only when the frozen rows are missing or explicitly bypassed
validation_rows_path = resolved_artifacts.get('validation_rows')
screen_summary_path = resolved_artifacts.get('screen_summary')
if screen_summary_path is None:
    raise FileNotFoundError('Block 2 needs the frozen candidate screen summary to define the regime.')
screen_summary = json.loads(screen_summary_path.read_text(encoding='utf-8'))
selected_regime = screen_summary.get('selected_regime', 'to_basic__AND__high_disagreement_q75')
selected_label = 'high-covariance / high-disagreement regime'
reuse_mode = validation_rows_path is not None and not FORCE_HEAVY
if reuse_mode:
    rows_df = pd.read_csv(validation_rows_path)
    rows_source = str(validation_rows_path)
else:
    raise RuntimeError(
        'Frozen validation rows were not found, and heavy rerun mode is intentionally disabled in this notebook draft. '
        'Set up the missing frozen bundle first or extend this notebook with the bounded rerun helper if you want live rescoring.'
    )
rows_df['selected_regime'] = rows_df.get('selected_regime', selected_regime).fillna(selected_regime)
rows_df['validation_role'] = rows_df.get('validation_role', 'unknown').fillna('unknown')
rows_df['label'] = rows_df.get('label', pd.Series(np.nan, index=rows_df.index)).fillna('unknown')
rows_df['label_binary'] = rows_df['label'].astype(str).str.lower().isin({'1', 'true', 'pathogenic', 'positive'}).astype(int)
rows_df['ll_norm_150m'] = rows_df.get('ll_norm', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['frob_norm_150m'] = rows_df.get('frob_norm', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['pair_norm_150m'] = rows_df.get('pair_norm', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['ll_norm_650m'] = rows_df.get('ll_norm_650m', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['frob_norm_650m'] = rows_df.get('frob_norm_650m', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['pair_norm_650m'] = rows_df.get('pair_norm_650m', pd.Series(np.nan, index=rows_df.index)).astype(float)
rows_df['frob_repair'] = rows_df['frob_norm_150m'] - rows_df['frob_norm_650m']
rows_df['pair_repair'] = rows_df['pair_norm_150m'] - rows_df['pair_norm_650m']
rows_df['ll_repair'] = rows_df['ll_norm_150m'] - rows_df['ll_norm_650m']
rows_df['variant_id'] = (
    rows_df['gene'].astype(str) + ':'
    + rows_df['wt_aa'].astype(str) + rows_df['position'].astype(str) + rows_df['mut_aa'].astype(str)
)
rows_df.to_csv(TABLES_DIR / 'failure_mode_validation_rows_reused_or_rerun.csv', index=False)
print({'rows_source': rows_source, 'row_count': int(len(rows_df)), 'reuse_mode': reuse_mode, 'selected_regime': selected_regime})
done('Validation rows are ready.')


In [ ]:
# Build the core Block 2 readout and exemplar ranking
candidate_mask = rows_df['validation_role'].astype(str).str.startswith('candidate_positive')
control_mask = rows_df['validation_role'].astype(str).eq('matched_positive_control')
benign_mask = rows_df['validation_role'].astype(str).eq('regime_benign')
repair_mask = (
    candidate_mask
    & (rows_df['frob_repair'] >= FROB_REPAIR_MIN)
    & (rows_df['pair_repair'] >= PAIR_REPAIR_MIN)
    & (rows_df['ll_repair'] <= LL_REPAIR_MAX)
)
rows_df['is_candidate'] = candidate_mask
rows_df['is_control'] = control_mask
rows_df['is_regime_benign'] = benign_mask
rows_df['is_repair_exemplar'] = repair_mask
candidate_df = rows_df.loc[candidate_mask].copy()
control_df = rows_df.loc[control_mask].copy()
repair_df = rows_df.loc[repair_mask].copy()
nonrepair_candidate_df = rows_df.loc[candidate_mask & ~repair_mask].copy()
core_readout = pd.DataFrame([
    {'metric': 'selected_regime', 'value': selected_regime},
    {'metric': 'selected_label', 'value': selected_label},
    {'metric': 'candidate_variant_count', 'value': int(candidate_df['variant_id'].nunique())},
    {'metric': 'control_variant_count', 'value': int(control_df['variant_id'].nunique())},
    {'metric': 'regime_benign_count', 'value': int(rows_df.loc[benign_mask, 'variant_id'].nunique())},
    {'metric': 'repair_exemplar_count', 'value': int(repair_df['variant_id'].nunique())},
    {'metric': 'repair_rate_candidates', 'value': float(repair_df['variant_id'].nunique() / max(candidate_df['variant_id'].nunique(), 1))},
    {'metric': 'candidate_frob_repair_mean', 'value': float(candidate_df['frob_repair'].mean())},
    {'metric': 'candidate_pair_repair_mean', 'value': float(candidate_df['pair_repair'].mean())},
    {'metric': 'candidate_ll_repair_mean', 'value': float(candidate_df['ll_repair'].mean())},
    {'metric': 'control_frob_repair_mean', 'value': float(control_df['frob_repair'].mean())},
    {'metric': 'control_pair_repair_mean', 'value': float(control_df['pair_repair'].mean())},
    {'metric': 'control_ll_repair_mean', 'value': float(control_df['ll_repair'].mean())},
])
core_readout.to_csv(TABLES_DIR / 'block2_core_readout.csv', index=False)
candidate_exemplars = repair_df.sort_values(['frob_repair', 'pair_repair', 'll_repair'], ascending=[False, False, True]).copy()
candidate_exemplars.to_csv(TABLES_DIR / 'candidate_exemplars_ranked.csv', index=False)
gene_scale_summary = candidate_df.groupby('gene', dropna=False).agg(
    candidate_count=('variant_id', 'nunique'),
    repair_exemplar_count=('is_repair_exemplar', 'sum'),
    frob_repair_mean=('frob_repair', 'mean'),
    pair_repair_mean=('pair_repair', 'mean'),
    ll_repair_mean=('ll_repair', 'mean'),
).reset_index().sort_values('frob_repair_mean', ascending=False)
gene_scale_summary.to_csv(TABLES_DIR / 'gene_scale_repair_summary.csv', index=False)
display(core_readout)
display(candidate_exemplars.head(10))
done('Core Block 2 readout saved.')


In [ ]:
# Annotate chemistry, AlphaFold pLDDT, and UniProt features with graceful fallbacks
uniprot_cache: dict[str, dict | None] = {}
alphafold_cache: dict[str, dict | None] = {}

def classify_substitution(row: pd.Series) -> dict:
    wt = str(row.get('wt_aa', ''))[:1]
    mut = str(row.get('mut_aa', ''))[:1]
    wt_class = AA_CHARGE.get(wt, 'unknown')
    mut_class = AA_CHARGE.get(mut, 'unknown')
    return {
        'to_basic': int(mut_class == 'basic'),
        'to_histidine': int(mut == 'H'),
        'to_arginine': int(mut == 'R'),
        'from_cysteine': int(wt == 'C'),
        'from_glycine': int(wt == 'G'),
        'from_proline': int(wt == 'P'),
        'from_aromatic': int(wt in {'F', 'W', 'Y'}),
        'from_special': int(wt in {'G', 'P', 'C'}),
        'charge_flip': int((wt_class == 'acidic' and mut_class == 'basic') or (wt_class == 'basic' and mut_class == 'acidic')),
        'large_volume_jump': int(abs(AA_VOLUME.get(wt, 0) - AA_VOLUME.get(mut, 0)) >= 3),
    }

def fetch_uniprot_entry(gene: str) -> dict | None:
    if gene in uniprot_cache:
        return uniprot_cache[gene]
    query = urllib.parse.quote(f'(gene_exact:{gene} AND organism_id:9606 AND reviewed:true)')
    payload = fetch_json(f'https://rest.uniprot.org/uniprotkb/search?query={query}&format=json&size=1')
    if not isinstance(payload, dict) or not payload.get('results'):
        uniprot_cache[gene] = None
        return None
    result = payload['results'][0]
    entry = {
        'primaryAccession': result.get('primaryAccession'),
        'features': result.get('features', []),
    }
    uniprot_cache[gene] = entry
    return entry

def fetch_alphafold_prediction(accession: str) -> dict | None:
    if accession in alphafold_cache:
        return alphafold_cache[accession]
    payload = fetch_json(f'https://alphafold.ebi.ac.uk/api/prediction/{accession}')
    if isinstance(payload, list) and payload:
        alphafold_cache[accession] = payload[0]
    else:
        alphafold_cache[accession] = None
    return alphafold_cache[accession]

def feature_overlap_flags(features: list[dict], position: int) -> dict:
    flags = {
        'in_domain': 0,
        'in_region': 0,
        'in_repeat': 0,
        'in_motif': 0,
        'in_binding_site': 0,
        'in_dna_binding': 0,
        'in_transmem': 0,
        'in_coiled': 0,
        'in_comp_bias': 0,
    }
    for feature in features or []:
        location = feature.get('location', {})
        start = safe_float(location.get('start', {}).get('value'), default=float('nan'))
        end = safe_float(location.get('end', {}).get('value'), default=float('nan'))
        if math.isnan(start) or math.isnan(end):
            continue
        if not (start <= position <= end):
            continue
        feature_type = str(feature.get('type', '')).lower()
        description = str(feature.get('description', '')).lower()
        if feature_type == 'domain':
            flags['in_domain'] = 1
        if feature_type == 'region':
            flags['in_region'] = 1
        if feature_type == 'repeat':
            flags['in_repeat'] = 1
        if feature_type == 'motif':
            flags['in_motif'] = 1
        if 'binding' in feature_type or 'binding' in description:
            flags['in_binding_site'] = 1
        if 'dna' in description:
            flags['in_dna_binding'] = 1
        if 'transmembrane' in feature_type or 'transmembrane' in description:
            flags['in_transmem'] = 1
        if 'coiled' in feature_type or 'coiled' in description:
            flags['in_coiled'] = 1
        if 'compositionally biased' in description or feature_type == 'compositionally biased region':
            flags['in_comp_bias'] = 1
    return flags

annotation_rows: list[dict] = []
for _, row in rows_df.iterrows():
    gene = str(row['gene'])
    position = int(safe_float(row.get('position'), default=-1))
    chemistry = classify_substitution(row)
    uniprot_entry = fetch_uniprot_entry(gene)
    accession = uniprot_entry.get('primaryAccession') if isinstance(uniprot_entry, dict) else None
    af_entry = fetch_alphafold_prediction(accession) if accession else None
    feature_flags = feature_overlap_flags(uniprot_entry.get('features', []) if isinstance(uniprot_entry, dict) else [], position)
    mean_plddt = safe_float(af_entry.get('modelConfidenceScore') if isinstance(af_entry, dict) else float('nan'))
    annotation_rows.append({
        'variant_id': row['variant_id'],
        'uniprot_accession': accession or '',
        'alphafold_prediction_available': int(af_entry is not None),
        'mean_plddt': mean_plddt,
        'structured_confident': int(not math.isnan(mean_plddt) and mean_plddt >= CONFIDENT_PLDDT),
        'very_high_confident': int(not math.isnan(mean_plddt) and mean_plddt >= VERY_HIGH_PLDDT),
        'disorder_like': int(not math.isnan(mean_plddt) and mean_plddt < DISORDER_PLDDT),
        **chemistry,
        **feature_flags,
    })
annotation_df = pd.DataFrame(annotation_rows)
rows_df = rows_df.merge(annotation_df, on='variant_id', how='left')
signature_parts = [
    'to_basic', 'charge_flip', 'from_aromatic', 'from_special',
    'structured_confident', 'in_domain', 'in_binding_site', 'in_dna_binding'
]
def build_signature(row: pd.Series) -> str:
    active = [name for name in signature_parts if int(safe_float(row.get(name), default=0)) == 1]
    return ';'.join(active) if active else 'unannotated'
rows_df['context_signature'] = rows_df.apply(build_signature, axis=1)
rows_df.to_csv(TABLES_DIR / 'annotated_failure_mode_rows.csv', index=False)
annotation_coverage = pd.DataFrame([
    {'metric': 'rows', 'value': int(len(rows_df))},
    {'metric': 'alphafold_prediction_available_fraction', 'value': float(rows_df['alphafold_prediction_available'].fillna(0).mean())},
    {'metric': 'mean_plddt_nonmissing_fraction', 'value': float(rows_df['mean_plddt'].notna().mean())},
    {'metric': 'unannotated_signature_fraction', 'value': float((rows_df['context_signature'] == 'unannotated').mean())},
])
annotation_coverage.to_csv(TABLES_DIR / 'annotation_coverage.csv', index=False)
display(annotation_coverage)
done('Context annotations saved.')


In [ ]:
# Summarize enrichment against controls, benign rows, and non-repair candidates
comparison_sets = {
    'repair_vs_controls': (rows_df.loc[repair_mask].copy(), rows_df.loc[control_mask].copy()),
    'repair_vs_regime_benign': (rows_df.loc[repair_mask].copy(), rows_df.loc[benign_mask].copy()),
    'repair_vs_nonrepair_candidates': (rows_df.loc[repair_mask].copy(), rows_df.loc[candidate_mask & ~repair_mask].copy()),
}
feature_cols = [
    'to_basic', 'to_histidine', 'to_arginine', 'from_cysteine', 'from_glycine', 'from_proline',
    'from_aromatic', 'from_special', 'charge_flip', 'large_volume_jump', 'structured_confident',
    'very_high_confident', 'disorder_like', 'in_domain', 'in_region', 'in_repeat', 'in_motif',
    'in_binding_site', 'in_dna_binding', 'in_transmem', 'in_coiled', 'in_comp_bias',
]
enrichment_rows: list[dict] = []
for comparison_name, (positive_df, background_df) in comparison_sets.items():
    for feature in feature_cols:
        pos_yes = int(positive_df.get(feature, pd.Series(dtype=float)).fillna(0).astype(int).sum())
        pos_no = int(len(positive_df) - pos_yes)
        bg_yes = int(background_df.get(feature, pd.Series(dtype=float)).fillna(0).astype(int).sum())
        bg_no = int(len(background_df) - bg_yes)
        odds_ratio, pvalue = fisher_exact([[pos_yes, pos_no], [bg_yes, bg_no]], alternative='greater') if len(positive_df) and len(background_df) else (float('nan'), float('nan'))
        pos_rate = pos_yes / max(len(positive_df), 1)
        bg_rate = bg_yes / max(len(background_df), 1)
        enrichment_rows.append({
            'comparison': comparison_name,
            'feature': feature,
            'positive_count': pos_yes,
            'positive_total': int(len(positive_df)),
            'background_count': bg_yes,
            'background_total': int(len(background_df)),
            'positive_rate': pos_rate,
            'background_rate': bg_rate,
            'rate_difference': pos_rate - bg_rate,
            'odds_ratio': float(odds_ratio),
            'pvalue_greater': float(pvalue),
        })
context_enrichment = pd.DataFrame(enrichment_rows).sort_values(['comparison', 'rate_difference', 'positive_count'], ascending=[True, False, False])
context_enrichment.to_csv(TABLES_DIR / 'context_enrichment_summary.csv', index=False)
signature_summary = rows_df.groupby(['validation_role', 'context_signature'], dropna=False).size().reset_index(name='count').sort_values(['validation_role', 'count'], ascending=[True, False])
signature_summary.to_csv(TABLES_DIR / 'context_signature_summary.csv', index=False)
supporting_rows: list[dict] = []
for key in ('scale_repair_summary', 'robustness_summary', 'sister_summary'):
    path = resolved_artifacts.get(key)
    if path is not None:
        payload = json.loads(path.read_text(encoding='utf-8'))
        for metric, value in payload.items():
            if isinstance(value, (str, int, float, bool)):
                supporting_rows.append({'source': key, 'metric': metric, 'value': value})
supporting_evidence = pd.DataFrame(supporting_rows)
supporting_evidence.to_csv(TABLES_DIR / 'supporting_evidence_summary.csv', index=False)
display(context_enrichment.head(20))
display(signature_summary.head(20))
done('Enrichment tables saved.')


In [ ]:
# Figures, bug statement, manifests, and zip bundle
plt.style.use('seaborn-v0_8-whitegrid')
gene_plot = gene_scale_summary.sort_values('frob_repair_mean', ascending=True)
fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(gene_plot))))
ax.barh(gene_plot['gene'], gene_plot['frob_repair_mean'], color='#0f766e', label='frob repair')
ax.barh(gene_plot['gene'], gene_plot['pair_repair_mean'], color='#f59e0b', alpha=0.65, label='pair repair')
ax.axvline(0.0, color='black', linewidth=1)
ax.set_xlabel('Gap reduction from 150M to 650M')
ax.set_title('Gene-level scale repair overview')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'gene_scale_repair_overview.png', dpi=200)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(candidate_df['frob_repair'], candidate_df['ll_repair'], c=candidate_df['pair_repair'], cmap='viridis', s=80, edgecolor='black')
ax.axvline(FROB_REPAIR_MIN, color='#0f766e', linestyle='--', linewidth=1)
ax.axhline(LL_REPAIR_MAX, color='#b91c1c', linestyle='--', linewidth=1)
ax.set_xlabel('Frob repair')
ax.set_ylabel('Likelihood repair')
ax.set_title('Candidate repair quadrants')
fig.colorbar(scatter, ax=ax, label='Pair repair')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'candidate_repair_quadrants.png', dpi=200)
plt.close(fig)
top_context = context_enrichment.loc[context_enrichment['comparison'].eq('repair_vs_controls')].sort_values(['rate_difference', 'positive_count'], ascending=[False, False]).head(10)
fig, ax = plt.subplots(figsize=(8, max(4, 0.4 * len(top_context))))
ax.barh(top_context['feature'], top_context['rate_difference'], color='#1d4ed8')
ax.axvline(0.0, color='black', linewidth=1)
ax.set_xlabel('Repair minus control rate')
ax.set_title('Top context enrichments vs matched controls')
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'context_enrichment_vs_controls.png', dpi=200)
plt.close(fig)
fig, ax = plt.subplots(figsize=(7, 5))
plot_df = rows_df.loc[rows_df['mean_plddt'].notna(), ['validation_role', 'mean_plddt']].copy()
if not plot_df.empty:
    order = list(plot_df.groupby('validation_role')['mean_plddt'].median().sort_values().index)
    data = [plot_df.loc[plot_df['validation_role'].eq(role), 'mean_plddt'].to_numpy() for role in order]
    ax.boxplot(data, labels=order, patch_artist=True)
    ax.set_ylabel('Mean AlphaFold confidence')
    ax.set_title('AlphaFold confidence by validation role')
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'plddt_by_validation_role.png', dpi=200)
    plt.close(fig)
else:
    plt.close(fig)
best_context = None
if not top_context.empty:
    eligible = top_context.loc[(top_context['rate_difference'] >= 0.20) & (top_context['positive_count'] >= MIN_CONTEXT_COUNT)]
    if not eligible.empty:
        best_context = eligible.iloc[0]
if best_context is not None:
    bug_statement = (
        f'SpectralBio isolates a covariance-heavy failure mode in {best_context.feature} substitutions: '
        f'the 150M backbone overstates the regime gap, the 650M backbone repairs it, and likelihood does not repair in parallel.'
    )
else:
    bug_statement = (
        'SpectralBio isolates a bounded scale-repair failure mode: the covariance-heavy disagreement regime is extreme at 150M, '
        'shrinks under 650M, and likelihood does not repair in parallel with the covariance channels.'
    )
summary = {
    'notebook_slug': NOTEBOOK_SLUG,
    'account_label': ACCOUNT_LABEL,
    'run_at': RUN_AT,
    'selected_regime': selected_regime,
    'selected_label': selected_label,
    'rows_source': rows_source,
    'reuse_mode': reuse_mode,
    'candidate_variant_count': int(candidate_df['variant_id'].nunique()),
    'control_variant_count': int(control_df['variant_id'].nunique()),
    'repair_exemplar_count': int(repair_df['variant_id'].nunique()),
    'repair_rate_candidates': float(repair_df['variant_id'].nunique() / max(candidate_df['variant_id'].nunique(), 1)),
    'bug_statement': bug_statement,
}
(MANIFESTS_DIR / 'block2_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
artifact_summary = {
    'tables_dir': str(TABLES_DIR),
    'figures_dir': str(FIGURES_DIR),
    'text_dir': str(TEXT_DIR),
    'zip_path': str(ZIP_PATH),
    'root_zip_copy': str(ROOT_ZIP_COPY),
}
(MANIFESTS_DIR / 'artifact_summary.json').write_text(json.dumps(artifact_summary, indent=2), encoding='utf-8')
summary_md = '\n'.join([
    '# Block 2 summary',
    '',
    f'- Selected regime: `{selected_regime}`',
    f'- Reviewer label: `{selected_label}`',
    f'- Rows source: `{rows_source}`',
    f'- Candidate variants: `{int(candidate_df["variant_id"].nunique())}`',
    f'- Matched controls: `{int(control_df["variant_id"].nunique())}`',
    f'- Repair exemplars: `{int(repair_df["variant_id"].nunique())}`',
    f'- Bug statement: {bug_statement}',
])
(TEXT_DIR / 'block2_summary.md').write_text(summary_md + '\n', encoding='utf-8')
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RESULTS_ROOT.rglob('*')):
        if path.is_file():
            bundle.write(path, arcname=path.relative_to(RESULTS_ROOT))
shutil.copy2(ZIP_PATH, ROOT_ZIP_COPY)
print(json.dumps(summary, indent=2))
print({'zip_path': str(ZIP_PATH), 'root_zip_copy': str(ROOT_ZIP_COPY)})
done('Block 2 bundle is ready for download.')
